In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# 데이터 불러오기
abalone_df = pd.read_csv(
    'https://storage.googleapis.com/download.tensorflow.org/data/abalone_train.csv',
    names=['Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight',
           'Viscera weight', 'Shell weight', 'Age']
)

# 입력과 타깃 나누기
input_data = abalone_df.drop(columns=['Age']).to_numpy().astype(np.float32)
target_data = abalone_df['Age'].to_numpy().astype(np.float32)

# 데이터셋 클래스 정의
class AbaloneDataset(Dataset):
    def __init__(self, input_data, target_data):
        self.input_data = input_data
        self.target_data = target_data

    def __len__(self):
        return len(self.input_data)

    def __getitem__(self, index):
        input_tensor = torch.tensor(self.input_data[index], dtype=torch.float32)
        target_tensor = torch.tensor(self.target_data[index], dtype=torch.float32)

        return input_tensor, target_tensor

# 학습/검증/테스트 분할
train_size = int(len(input_data) * 0.8)
val_size = int(len(input_data) * 0.1)

train_inputs = input_data[:train_size]
train_targets = target_data[:train_size]

val_inputs = input_data[train_size:train_size + val_size]
val_targets = target_data[train_size:train_size + val_size]

test_inputs = input_data[train_size + val_size:]
test_targets = target_data[train_size + val_size:]

# 표준화
scaler = StandardScaler()
scaler.fit(train_inputs)

train_inputs_scaled = scaler.transform(train_inputs)
val_inputs_scaled = scaler.transform(val_inputs)
test_inputs_scaled = scaler.transform(test_inputs)

# 데이터셋 만들기
train_dataset = AbaloneDataset(train_inputs_scaled, train_targets)
val_dataset = AbaloneDataset(val_inputs_scaled, val_targets)
test_dataset = AbaloneDataset(test_inputs_scaled, test_targets)

# 데이터로더 만들기
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True, drop_last=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=32)

In [ ]:
# 모델 클래스 정의
class AbaloneModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(7, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 8)
        self.fc4 = nn.Linear(8, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

model = AbaloneModel()

In [ ]:
loss_fn = nn.MSELoss()

In [ ]:
epochs = 1
for epoch in range(epochs):
    for train_batch in train_dataloader:
        x_train, y_train = train_batch
        pred = model(x_train).squeeze()
        loss = loss_fn(pred, y_train)

        #loss를 통해 gradient 계산
        loss.backward()
        # gradient를 활용해 모델 파라미터 업데이트
        break

In [ ]:
for name, param in model.named_parameters():
    print(f'{name}: {param.grad}')

fc1.weight: tensor([[-0.0830, -0.0862, -0.0499, -0.0905, -0.0915, -0.0804, -0.0917],
        [ 0.1387,  0.1376,  0.0912,  0.1562,  0.1591,  0.1444,  0.1321],
        [-0.1967, -0.1848, -0.1283, -0.2252, -0.2292, -0.2182, -0.1786],
        [-0.1304, -0.1140, -0.1523, -0.1687, -0.1304, -0.1832, -0.1202],
        [ 0.0609,  0.0594,  0.0320,  0.0746,  0.0666,  0.0569,  0.0708],
        [-0.0578, -0.0587, -0.0257, -0.0576, -0.0597, -0.0331, -0.0756],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0474, -0.0371, -0.0159, -0.0133, -0.0238, -0.0382, -0.0095],
        [ 0.0920,  0.0864,  0.0519,  0.1054,  0.1050,  0.1004,  0.0805],
        [-0.0450, -0.0480,  0.0120, -0.0286, -0.0560, -0.0029, -0.0535],
        [ 0.0166,  0.0207, -0.0067,  0.0331,  0.0369,  0.0263,  0.0333],
        [ 0.0297,  0.0277,  0.0385,  0.0273,  0.0313,  0.0293,  0.0295],
        [ 0.0089,  0.0040, -0.0085,  0.0004,  0.0071, -0.0014, -0.0027],
        [-0.0797, -0.0796, -0.0444, -0.

In [ ]:
x = torch.tensor([2.0])
x.requires_grad

False

In [ ]:
x.requires_grad_(False)
x.requires_grad

False

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 3
y

tensor([8.], grad_fn=<PowBackward0>)

In [ ]:
y.backward()
x.grad

tensor([12.])

$$y=x^3$$
$$y'=3x^2$$
$$y' = 3 * 2^2 = 12$$

In [ ]:
x = torch.tensor([2.0], requires_grad=False)
y = x ** 3
y

tensor([8.])

In [ ]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn